```
Clone original rnnoise repository and helper scripts from ML-zoo
```

In [ ]:
!git clone -b 0.1.x https://github.com/xiph/rnnoise.git
%cd rnnoise
!wget -O model.py \
https://raw.githubusercontent.com/ARM-software/ML-zoo/master/models/noise_suppression/RNNoise/tflite_int8/recreate_model/model.py

!wget -O data.py \
https://raw.githubusercontent.com/ARM-software/ML-zoo/master/models/noise_suppression/RNNoise/tflite_int8/recreate_model/data.py

!wget -O rnnoise_pre_processing.py \
https://raw.githubusercontent.com/ARM-software/ML-zoo/master/models/noise_suppression/RNNoise/tflite_int8/recreate_model/rnnoise_pre_processing.py


Cloning into 'rnnoise'...
remote: Enumerating objects: 907, done.
remote: Counting objects: 100% (426/426), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 907 (delta 383), reused 349 (delta 349), pack-reused 481 (from 2)
Receiving objects: 100% (907/907), 1.02 MiB | 13.40 MiB/s, done.
Resolving deltas: 100% (502/502), done.
/content/rnnoise
--2026-06-12 08:27:07--  https://raw.githubusercontent.com/ARM-software/ML-zoo/master/models/noise_suppression/RNNoise/tflite_int8/recreate_model/model.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5382 (5.3K) [text/plain]
Saving to: ‘model.py’

model.py            100%[===================>]   5.26K  --.-KB/s    in 0s      

2026-06-12 08:27:07 (73.1 MB/s) - ‘model.py’ saved [5382/5382]

--2026-

```
Update outdate scripts
```

In [ ]:
import re

file = "/content/rnnoise/model.py"

with open(file, "r") as f:
    txt = f.read()

# change shape=24 / shape = 24
txt = re.sub(r"shape\s*=\s*24\b", "shape=(24,)", txt)
txt = re.sub(r"shape\s*=\s*48\b", "shape=(48,)", txt)
txt = re.sub(r"shape\s*=\s*96\b", "shape=(96,)", txt)
txt = re.sub(r"shape\s*=\s*42\b", "shape=(42,)", txt)

# change Input(24, ...)
txt = re.sub(r"Input\(\s*24\s*,", "Input(shape=(24,),", txt)
txt = re.sub(r"Input\(\s*48\s*,", "Input(shape=(48,),", txt)
txt = re.sub(r"Input\(\s*96\s*,", "Input(shape=(96,),", txt)
txt = re.sub(r"Input\(\s*42\s*,", "Input(shape=(42,),", txt)

with open(file, "w") as f:
    f.write(txt)

print("Updated")

patched


```
Update and compile for denoise_training executable file
```

In [ ]:
!apt update -qq
!apt install -qq -y autoconf automake libtool build-essential pkg-config parallel ffmpeg

76 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
autoconf is already the newest version (2.71-2).
autoconf set to manually installed.
automake is already the newest version (1:1.16.5-1.3).
automake set to manually installed.
build-essential is already the newest version (12.9ubuntu3).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
The following packages were automatically installed and are no longer required:
  libbz2-dev libpkgconf3 libreadline-dev
Use 'apt autoremove' to remove them.
The following additional packages will be installed:
  sysstat
Suggested packages:
  libtool-doc gcj-jdk ash csh fish ksh tcsh zsh isag
The following packages will be REMOVED:
  pkgconf r-base-dev
The following NEW packages will be installed:
  libtool parallel pkg-config sysstat


In [ ]:
!./autogen.sh && ./configure && make

Updating build configuration files for rnnoise, please wait....
libtoolize: putting auxiliary files in '.'.
libtoolize: linking file './ltmain.sh'
libtoolize: putting macros in AC_CONFIG_MACRO_DIRS, 'm4'.
libtoolize: linking file 'm4/libtool.m4'
libtoolize: linking file 'm4/ltoptions.m4'
libtoolize: linking file 'm4/ltsugar.m4'
libtoolize: linking file 'm4/ltversion.m4'
libtoolize: linking file 'm4/lt~obsolete.m4'
configure.ac:19: installing './compile'
configure.ac:27: installing './config.guess'
configure.ac:27: installing './config.sub'
configure.ac:22: installing './install-sh'
configure.ac:22: installing './missing'
Makefile.am: installing './depcomp'
checking for gcc... gcc
checking whether the C compiler works... yes
checking for C compiler default output file name... a.out
checking for suffix of executables... 
checking whether we are cross compiling... no
checking for suffix of object files... o
checking whether the compiler supports GNU C... yes
checking whether gcc accepts -

In [ ]:
%cd ./src
!./compile.sh

/content/rnnoise/src
denoise.c: In function ‘main’:
denoise.c:545:5: warning: ignoring return value of ‘fread’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
  545 |     fread(tmp, sizeof(short), FRAME_SIZE, f2);
      |     ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
denoise.c:577:7: warning: ignoring return value of ‘fread’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
  577 |       fread(tmp, sizeof(short), FRAME_SIZE, f1);
      |       ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
denoise.c:580:9: warning: ignoring return value of ‘fread’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
  580 |         fread(tmp, sizeof(short), FRAME_SIZE, f1);
      |         ^~~~~~~~~~~~~~~~~~~~~

```
Download Speech Training Dataset
```

In [ ]:
!curl -L -O https://datashare.ed.ac.uk/bitstream/handle/10283/2791/clean_testset_wav.zip
!curl -L -O https://datashare.ed.ac.uk/bitstream/handle/10283/2791/clean_trainset_56spk_wav.zip
!curl -L -O https://datashare.ed.ac.uk/bitstream/handle/10283/2791/noisy_testset_wav.zip
!curl -L -O https://datashare.ed.ac.uk/bitstream/handle/10283/2791/noisy_trainset_56spk_wav.zip

!unzip -q clean_testset_wav.zip
!unzip -q noisy_testset_wav.zip
!unzip -q clean_trainset_56spk_wav.zip
!unzip -q noisy_trainset_56spk_wav.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  147M  100  147M    0     0   866k      0  0:02:53  0:02:53 --:--:-- 1421k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 4549M  100 4549M    0     0  12.4M      0  0:06:06  0:06:06 --:--:-- 13.5M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  162M  100  162M    0     0  1155k      0  0:02:24  0:02:24 --:--:-- 1763k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 5366M  100 5366M    0     0  7749k      0  0:11:49  0:11:49 --:--:-- 9344k


```
Create speech and noisy PCM format file for training and testing
```

In [ ]:
import librosa
import numpy as np
from pathlib import Path

def generate_pcm(clean_dir, noisy_dir, speech_pcm, noise_pcm):
    clean_dir = Path(clean_dir)
    noisy_dir = Path(noisy_dir)

    with open(speech_pcm, "wb") as speech_out, open(noise_pcm, "wb") as noise_out:
        for clean_file in clean_dir.rglob("*.wav"):
            noisy_file = noisy_dir / clean_file.name

            if not noisy_file.exists():
                print(f"Skip: {clean_file.name}")
                continue

            clean, _ = librosa.load(clean_file, sr=48000, mono=True)
            noisy, _ = librosa.load(noisy_file, sr=48000, mono=True)

            n = min(len(clean), len(noisy))
            clean = clean[:n]
            noise = noisy[:n] - clean[:n]

            speech_i16 = np.clip(
                clean * 32768,
                -32768,
                32767
            ).astype(np.int16)

            noise_i16 = np.clip(
                noise * 32768,
                -32768,
                32767
            ).astype(np.int16)

            speech_i16.tofile(speech_out)
            noise_i16.tofile(noise_out)

    print(f"Generated: {speech_pcm}, {noise_pcm}")


# Training dataset
generate_pcm(
    "clean_trainset_56spk_wav",
    "noisy_trainset_56spk_wav",
    "speech.pcm",
    "noise.pcm"
)

# Test dataset
generate_pcm(
    "clean_testset_wav",
    "noisy_testset_wav",
    "speech_test.pcm",
    "noise_test.pcm"
)

Generated: speech.pcm, noise.pcm
Generated: speech_test.pcm, noise_test.pcm


```
Create train.f32 and test.f32 file
```

In [ ]:
!./denoise_training speech.pcm noise.pcm 500000 > train.f32
!./denoise_training speech_test.pcm noise_test.pcm 500000 > test.f32

matrix size: 500000 x 87
matrix size: 500000 x 87


```
Create train and test .h files
```

In [ ]:
!python3 ./../training/bin2hdf5.py ./train.f32 500000 87 ./train.h5
!python3 ./../training/bin2hdf5.py ./test.f32 500000 87 ./test.h5

```
Start train and convert stage. These steps below equivalent to training_and_quantise_model.sh script
```

In [ ]:
import sys

import logging
import tensorflow as tf

sys.path.insert(0,"/content/rnnoise")

from data import get_tf_dataset_from_h5
from model import rnnoise_model


def my_crossentropy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1.0 - 1e-7)

    bce = tf.expand_dims(
        tf.keras.losses.binary_crossentropy(y_true, y_pred),
        axis=-1
    )

    weight = 2.0 * tf.abs(y_true - 0.5)
    return tf.reduce_mean(weight * bce, axis=-1)


def my_mask(y_true):
    return tf.minimum(y_true + 1.0, 1.0)


def msse(y_true, y_pred):
    y_true = tf.clip_by_value(y_true, 1e-7, 1.0)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)

    return tf.reduce_mean(
        my_mask(y_true) * tf.square(tf.sqrt(y_pred) - tf.sqrt(y_true)),
        axis=-1
    )


def my_cost(y_true, y_pred):
    y_true = tf.clip_by_value(y_true, 1e-7, 1.0)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)

    diff = tf.sqrt(y_pred) - tf.sqrt(y_true)

    bce = tf.expand_dims(
        tf.keras.losses.binary_crossentropy(y_true, y_pred),
        axis=-1
    )

    return tf.reduce_mean(
        my_mask(y_true) *
        (
            10.0 * tf.square(tf.square(diff)) +
            tf.square(diff) +
            0.01 * bce
        ),
        axis=-1
    )


def train_colab(
    train_data_h5="train.h5",
    test_data_h5="test.h5",
    epochs=120,
    window_size=2000,
    batch_size=32,
):
    ckpt_path = "./ckpts/{epoch}.weights.h5"
    logs_path = "./logs"

    x_train = get_tf_dataset_from_h5(
        train_data_h5,
        window_size,
        batch_size
    )

    x_test = get_tf_dataset_from_h5(
        test_data_h5,
        window_size,
        batch_size
    )

    model = rnnoise_model(
        window_size,
        None,
        is_training=True
    )

    model.compile(
        loss=[my_cost, my_crossentropy],
        metrics=[msse, "accuracy"],
        optimizer="adam",
        loss_weights=[10.0, 0.5],
    )

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=ckpt_path,
            save_weights_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.TensorBoard(log_dir=logs_path),
    ]

    model.fit(
        x_train,
        validation_data=x_test,
        epochs=epochs,
        callbacks=callbacks,
        verbose=2,
    )

    logging.info("Training complete.")
    return model


logging.basicConfig(level=logging.INFO)

model = train_colab(
    train_data_h5="train.h5",
    test_data_h5="test.h5",
    epochs=120,
    window_size=2000,
    batch_size=32,
)

Epoch 1/120

Epoch 1: saving model to ./ckpts/1.weights.h5

Epoch 1: finished saving model to ./ckpts/1.weights.h5
7/7 - 41s - 6s/step - denoise_output_loss: 1.3809 - denoise_output_msse: 0.2497 - loss: 14.1989 - vad_output_accuracy: 0.3632 - vad_output_loss: 0.7781 - val_denoise_output_loss: 1.0113 - val_denoise_output_msse: 0.2123 - val_loss: 10.4583 - val_vad_output_accuracy: 0.3589 - val_vad_output_loss: 0.6894
Epoch 2/120

Epoch 2: saving model to ./ckpts/2.weights.h5

Epoch 2: finished saving model to ./ckpts/2.weights.h5
7/7 - 36s - 5s/step - denoise_output_loss: 0.8603 - denoise_output_msse: 0.1983 - loss: 8.9221 - vad_output_accuracy: 0.3978 - vad_output_loss: 0.6371 - val_denoise_output_loss: 0.6927 - val_denoise_output_msse: 0.1764 - val_loss: 7.2276 - val_vad_output_accuracy: 0.5005 - val_vad_output_loss: 0.6011
Epoch 3/120

Epoch 3: saving model to ./ckpts/3.weights.h5

Epoch 3: finished saving model to ./ckpts/3.weights.h5
7/7 - 32s - 5s/step - denoise_output_loss: 0.6388

In [ ]:
import sys
import os
import logging
import numpy as np
import tensorflow as tf

sys.path.insert(0, "/content/rnnoise")

from model import rnnoise_model_tflite
from data import get_tf_dataset_from_h5

NUM_CALIB = 1000


def normalize_ckpt_path(path):
    if path.endswith(".weights.h5"):
        return path

    candidate = path + ".weights.h5"
    if os.path.exists(candidate):
        return candidate

    return path


def load_model_for_tflite(ckpt_path):
    model = rnnoise_model_tflite(timesteps=1)

    ckpt_path = normalize_ckpt_path(ckpt_path)

    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

    model.load_weights(ckpt_path)
    return model


def collect_calibration_data(data_h5_path, ckpt_path):
    ds_calib = get_tf_dataset_from_h5(
        data_h5_path,
        window_size=1,
        batch_size=1
    ).take(NUM_CALIB)

    model = load_model_for_tflite(ckpt_path)

    vad_gru_state = np.zeros((1, 24), dtype=np.float32)
    noise_gru_state = np.zeros((1, 48), dtype=np.float32)
    denoise_gru_state = np.zeros((1, 96), dtype=np.float32)

    calibration_set = []

    for data in ds_calib:
        input_tensor = data[0].numpy().astype(np.float32)

        calibration_set.append(
            (
                input_tensor,
                vad_gru_state.copy(),
                noise_gru_state.copy(),
                denoise_gru_state.copy(),
            )
        )

        outputs = model(
            {
                "main_input": input_tensor,
                "vad_gru_prev_state": vad_gru_state,
                "noise_gru_prev_state": noise_gru_state,
                "denoise_gru_prev_state": denoise_gru_state,
            },
            training=False,
        )

        _, _, vad_state_out, noise_state_out, denoise_state_out = outputs

        vad_gru_state = vad_state_out.numpy()
        noise_gru_state = noise_state_out.numpy()
        denoise_gru_state = denoise_state_out.numpy()

        if vad_gru_state.ndim == 3:
            vad_gru_state = np.squeeze(vad_gru_state, axis=1)
        if noise_gru_state.ndim == 3:
            noise_gru_state = np.squeeze(noise_gru_state, axis=1)
        if denoise_gru_state.ndim == 3:
            denoise_gru_state = np.squeeze(denoise_gru_state, axis=1)

    return calibration_set


def convert_fp32(ckpt_path, output_path):
    model = load_model_for_tflite(ckpt_path)

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()

    with open(output_path, "wb") as f:
        f.write(tflite_model)

    logging.info("FP32 TFLite model generated: %s", output_path)


def convert_int8(ckpt_path, h5_path, output_path):
    model = load_model_for_tflite(ckpt_path)

    calibration_data = collect_calibration_data(h5_path, ckpt_path)

    def representative_dataset():
        for input_tensor, vad_gru, noise_gru, denoise_gru in calibration_data:
            yield {
                "main_input": input_tensor.astype(np.float32),
                "vad_gru_prev_state": vad_gru.astype(np.float32),
                "noise_gru_prev_state": noise_gru.astype(np.float32),
                "denoise_gru_prev_state": denoise_gru.astype(np.float32),
            }

    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS_INT8
    ]

    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_model = converter.convert()

    with open(output_path, "wb") as f:
        f.write(tflite_model)

    logging.info("INT8 TFLite model generated: %s", output_path)


def convert_colab(
    ckpt_path="/content/rnnoise/src/ckpts/120.weights.h5",
    h5_path="/content/rnnoise/src/train.h5",
    fp32_output="/content/rnnoise/rnnoise_1_step_fp32.tflite",
    int8_output="/content/rnnoise/rnnoise_1_step_int8.tflite",
):
    ckpt_path = normalize_ckpt_path(ckpt_path)

    convert_fp32(
        ckpt_path,
        fp32_output
    )

    convert_int8(
        ckpt_path,
        h5_path,
        int8_output
    )


logging.basicConfig(level=logging.INFO)

convert_colab(
    ckpt_path="/content/rnnoise/src/ckpts/10.weights.h5",
    h5_path="/content/rnnoise/src/train.h5",
    fp32_output="/content/rnnoise/rnnoise_fp32.tflite",
    int8_output="/content/rnnoise/rnnoise_int8.tflite",
)

Saved artifact at '/tmp/tmpuv98g10f'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(1, 1, 42), dtype=tf.float32, name='main_input'), TensorSpec(shape=(1, 24), dtype=tf.float32, name='vad_gru_prev_state'), TensorSpec(shape=(1, 48), dtype=tf.float32, name='noise_gru_prev_state'), TensorSpec(shape=(1, 96), dtype=tf.float32, name='denoise_gru_prev_state')]
Output Type:
  List[TensorSpec(shape=(1, 1, 22), dtype=tf.float32, name=None), TensorSpec(shape=(1, 1, 1), dtype=tf.float32, name=None), TensorSpec(shape=(1, 1, 24), dtype=tf.float32, name=None), TensorSpec(shape=(1, 1, 48), dtype=tf.float32, name=None), TensorSpec(shape=(1, 1, 96), dtype=tf.float32, name=None)]
Captures:
  134798182653712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134798182655056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134798182651600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134798182658320: TensorSpec(shape=(), dtype=t

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


```
Confirm .tflite files created
```

In [ ]:
!ls -lh /content/rnnoise/*.tflite

-rw-r--r-- 1 root root 359K Jun 12 10:48 /content/rnnoise/rnnoise_1_step_fp32.tflite
-rw-r--r-- 1 root root 133K Jun 12 10:49 /content/rnnoise/rnnoise_1_step_int8.tflite
-rw-r--r-- 1 root root 359K Jun 12 10:49 /content/rnnoise/rnnoise_fp32.tflite
-rw-r--r-- 1 root root 133K Jun 12 10:50 /content/rnnoise/rnnoise_int8.tflite
